# 🏆 Predict Customer Churn — Kaggle PS S6E3
### LightGBM + Target Encoding + Optuna  
**Hedef:** ROC-AUC

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED     = 42
N_FOLDS  = 10
N_TRIALS = 50
SEEDS    = [42, 2024, 777]   # hız için 3 seed

np.random.seed(SEED)
print('✅ Libraries loaded')

In [ ]:
# VERİ YÜKLEME 
if os.path.exists('/kaggle/input/playground-series-s6e3'):
    PATH = '/kaggle/input/playground-series-s6e3/'
else:
    PATH = './'

train = pd.read_csv(PATH + 'train.csv')
test  = pd.read_csv(PATH + 'test.csv')
sub   = pd.read_csv(PATH + 'sample_submission.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')

In [ ]:
#  FEATURE ENGINEERING 
def base_features(df):
    df = df.copy()

    # TotalCharges fix
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)

    # Numerik türevler
    df['AvgMonthlySpend']     = df['TotalCharges'] / (df['tenure'] + 1)
    df['MonthlyToTotalRatio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
    df['TotalChargesLog']     = np.log1p(df['TotalCharges'])
    df['MonthlyChargesLog']   = np.log1p(df['MonthlyCharges'])
    df['TenureLog']           = np.log1p(df['tenure'])
    df['ChargePerMonth']      = np.where(df['tenure'] > 0,
                                         df['TotalCharges'] / df['tenure'], 0)
    df['TenureMonthlyProduct']= df['tenure'] * df['MonthlyCharges']
    df['TenureSquared']       = df['tenure'] ** 2
    df['MonthlyChargesSq']    = df['MonthlyCharges'] ** 2
    df['DiffMonthlyAvg']      = df['MonthlyCharges'] - df['AvgMonthlySpend']

    # Servis bayrakları
    services = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
    for col in services:
        df[col + '_bin'] = (df[col] == 'Yes').astype(int)
    df['NumServices']         = df[[c + '_bin' for c in services]].sum(axis=1)
    df['NumSecuritySvcs']     = (df['OnlineSecurity_bin'] + df['OnlineBackup_bin'] +
                                  df['DeviceProtection_bin'] + df['TechSupport_bin'])
    df['NumStreamingSvcs']    = df['StreamingTV_bin'] + df['StreamingMovies_bin']

    # Internet tipi
    df['HasInternet']         = (df['InternetService'] != 'No').astype(int)
    df['FiberOptic']          = (df['InternetService'] == 'Fiber optic').astype(int)

    # Ödeme
    df['ElectronicCheck']     = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['AutoPayment']         = df['PaymentMethod'].isin(
                                    ['Bank transfer (automatic)',
                                     'Credit card (automatic)']).astype(int)

    # Sözleşme
    df['ContractOrdinal']     = df['Contract'].map(
                                    {'Month-to-month': 0, 'One year': 1, 'Two year': 2})
    df['PaperlessBilling_bin']= (df['PaperlessBilling'] == 'Yes').astype(int)

    # Demografik
    df['Gender_bin']          = (df['gender'] == 'Male').astype(int)
    df['Partner_bin']         = (df['Partner'] == 'Yes').astype(int)
    df['Dependents_bin']      = (df['Dependents'] == 'Yes').astype(int)
    df['SeniorNoPartner']     = df['SeniorCitizen'] * (df['Partner'] == 'No').astype(int)

    # Etkileşimler
    df['HighRiskProfile']     = ((df['ContractOrdinal'] == 0) &
                                  (df['ElectronicCheck'] == 1) &
                                  (df['FiberOptic'] == 1)).astype(int)
    df['ContractXTenure']     = df['ContractOrdinal'] * df['tenure']
    df['FiberXMonthly']       = df['FiberOptic'] * df['MonthlyCharges']
    df['NoSupportNoSecurity'] = ((df['NumSecuritySvcs'] == 0) &
                                  df['HasInternet']).astype(int)
    df['LoyalCustomer']       = ((df['ContractOrdinal'] == 2) &
                                  (df['tenure'] > 24)).astype(int)
    df['NumSvcs_x_Monthly']   = df['NumServices'] * df['MonthlyCharges']

    return df

train = base_features(train)
test  = base_features(test)
print(f'✅ Feature engineering done | Shape: {train.shape}')

In [ ]:
#  LABEL ENCODING 
TARGET   = 'Churn'
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
            'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod']

le = LabelEncoder()
for col in cat_cols:
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

# Churn → 0/1
if train[TARGET].dtype == object:
    train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0})

FEATURES = [c for c in train.columns if c not in ['id', TARGET]]
X        = train[FEATURES].reset_index(drop=True)
y        = train[TARGET].reset_index(drop=True)
X_test   = test[FEATURES].reset_index(drop=True)

print(f'Features: {len(FEATURES)} | Churn rate: {y.mean():.4f}')

In [ ]:
# TARGET ENCODING (CV ile leakage önlenir) 
# Orijinal kategorik kolonları target encode et
te_cols = ['InternetService', 'Contract', 'PaymentMethod',
           'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
           'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

# Encode edilmiş hali (0,1,2...) zaten var, isimleri eşleştir
te_feature_names = [c + '_te' for c in te_cols]

global_mean = y.mean()

te_train = pd.DataFrame(index=X.index)
te_test  = pd.DataFrame(index=X_test.index)

cv_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for col in te_cols:
    te_train[col + '_te'] = 0.0
    # OOF target encoding
    for tr_idx, val_idx in cv_te.split(X, y):
        means = y.iloc[tr_idx].groupby(X[col].iloc[tr_idx]).mean()
        te_train.loc[val_idx, col + '_te'] = X[col].iloc[val_idx].map(means).fillna(global_mean).values
    # Test: tüm train ile encode
    means_full = y.groupby(X[col]).mean()
    te_test[col + '_te'] = X_test[col].map(means_full).fillna(global_mean).values

X      = pd.concat([X.reset_index(drop=True),      te_train.reset_index(drop=True)], axis=1)
X_test = pd.concat([X_test.reset_index(drop=True), te_test.reset_index(drop=True)],  axis=1)

FEATURES = list(X.columns)
print(f'✅ Target encoding done | Total features: {len(FEATURES)}')

In [ ]:
# OPTUNA — LightGBM 
def objective(trial):
    params = dict(
        objective        = 'binary',
        metric           = 'auc',
        verbosity        = -1,
        boosting_type    = trial.suggest_categorical('boosting_type', ['gbdt', 'dart']),
        n_estimators     = trial.suggest_int('n_estimators', 500, 3000),
        learning_rate    = trial.suggest_float('learning_rate', 0.005, 0.08, log=True),
        max_depth        = trial.suggest_int('max_depth', 4, 12),
        num_leaves       = trial.suggest_int('num_leaves', 31, 512),
        min_child_samples= trial.suggest_int('min_child_samples', 10, 200),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        subsample_freq   = trial.suggest_int('subsample_freq', 1, 10),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        min_split_gain   = trial.suggest_float('min_split_gain', 0, 1.0),
        random_state     = SEED,
        n_jobs           = -1,
    )
    cv     = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    for tr_idx, val_idx in cv.split(X, y):
        Xtr, Xval = X.iloc[tr_idx], X.iloc[val_idx]
        ytr, yval = y.iloc[tr_idx], y.iloc[val_idx]
        m = lgb.LGBMClassifier(**params)
        m.fit(Xtr, ytr, eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])
        scores.append(roc_auc_score(yval, m.predict_proba(Xval)[:, 1]))
    return np.mean(scores)

study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params
print(f'\n✅ Best Optuna AUC : {study.best_value:.6f}')
print(f'   Best params     : {best_params}')

In [ ]:
# FINAL CV + SEED AVERAGING
final_params = dict(
    **best_params,
    objective    = 'binary',
    metric       = 'auc',
    verbosity    = -1,
    n_jobs       = -1,
)

all_oof  = np.zeros(len(X))
all_test = np.zeros(len(X_test))

for seed in SEEDS:
    oof_s  = np.zeros(len(X))
    test_s = np.zeros(len(X_test))
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y), 1):
        Xtr, Xval = X.iloc[tr_idx], X.iloc[val_idx]
        ytr, yval = y.iloc[tr_idx], y.iloc[val_idx]

        m = lgb.LGBMClassifier(**{**final_params, 'random_state': seed})
        m.fit(Xtr, ytr, eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(100, verbose=False),
                         lgb.log_evaluation(-1)])

        oof_s[val_idx] = m.predict_proba(Xval)[:, 1]
        test_s        += m.predict_proba(X_test)[:, 1] / N_FOLDS

        print(f'  Seed {seed} | Fold {fold:02d}: {roc_auc_score(yval, oof_s[val_idx]):.6f}')

    seed_auc = roc_auc_score(y, oof_s)
    print(f'  → Seed {seed} OOF AUC: {seed_auc:.6f}\n')

    all_oof  += oof_s  / len(SEEDS)
    all_test += test_s / len(SEEDS)

final_auc = roc_auc_score(y, all_oof)
print(f'🏆 FINAL OOF AUC (seed avg): {final_auc:.6f}')

In [ ]:
# SUBMISSION
sub['Churn'] = all_test
sub.to_csv('submission.csv', index=False)
print('✅ submission.csv kaydedildi')
print(sub.head())